###

In [7]:
import json
import os
import csv
import random
import shutil
from pathlib import Path
from collections import defaultdict
from multiprocessing import Pool
import numpy as np

import cv2
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

In [8]:
TRAIN_IMG_DIR  = r"C:\Users\A\Desktop\307.생활폐기물 데이터 활용ㆍ환류\01-1.정식개방데이터\Training\01.원천데이터"
TRAIN_JSON_DIR = r"C:\Users\A\Desktop\307.생활폐기물 데이터 활용ㆍ환류\01-1.정식개방데이터\Training\02.라벨링데이터"
VAL_IMG_DIR    = r"C:\Users\A\Desktop\307.생활폐기물 데이터 활용ㆍ환류\01-1.정식개방데이터\Validation\01.원천데이터"
VAL_JSON_DIR   = r"C:\Users\A\Desktop\307.생활폐기물 데이터 활용ㆍ환류\01-1.정식개방데이터\Validation\02.라벨링데이터"

OUTPUT_DIR   = "yolo_dataset"
SUBSET_SIZE  = 55000
RANDOM_SEED  = 42

CLASS_MAP = {
    "c_1": 0, "c_1_01": 0,
    "c_2_01": 1,
    "c_2_02": 2, "c_2_02_01": 2,
    "c_3": 3, "c_3_01": 3,
    "c_4_01_02": 4, "c_4_01_01": 4,
    "c_4_02_01_02": 5, "c_4_02_02_02": 5,
    "c_4_02_03_02": 5, "c_4_03": 5,
    "c_4_03_01": 5, "c_4_02_01_01": 5,
    "c_4_02_02_01": 5, "c_4_02_03_01": 5,
    "c_5_02": 6, "c_5_01": 6,
    "c_5_01_01": 6, "c_5_02_01": 6,
    "c_6": 7, "c_6_01": 7,
    "c_7": 8, "c_7_01": 8,
    "c_8_01": 9, "c_8_02": 9, "c_8_01_01": 9,
    "c_9": 10,
}

print("설정 완료")

설정 완료


In [9]:
def scan_dataset(json_dir, img_dir):
    # 이미지 인덱싱
    img_index = {}
    for img_path in Path(img_dir).rglob("*.jpg"):
        img_index[img_path.stem] = str(img_path)
    for img_path in Path(img_dir).rglob("*.png"):
        img_index[img_path.stem] = str(img_path)
    print(f"이미지 {len(img_index)}개 인덱싱 완료")

    json_files = list(Path(json_dir).rglob("*.json"))
    print(f"JSON {len(json_files)}개 발견")

    rows = []
    for json_path in tqdm(json_files, desc="스캔 중"):
        try:
            with open(json_path, encoding="utf-8") as f:
                data = json.load(f)

            stem = json_path.stem
            if stem not in img_index:
                continue

            res   = data["Info"]["RESOLUTION"].split("/")
            img_w = int(res[0])
            img_h = int(res[1])

            classes = [
                obj["class_name"]
                for obj in data["objects"]
                if obj["class_name"] in CLASS_MAP
            ]
            if not classes:
                continue

            rows.append({
                "json_path": str(json_path),
                "img_path":  img_index[stem],
                "img_w":     img_w,
                "img_h":     img_h,
                "classes":   ",".join(classes),
            })

        except Exception as e:
            print(f"스킵: {json_path} ({e})")
            continue

    return rows

print("함수 정의 완료")

함수 정의 완료


In [10]:
print("Training scanning...")
train_rows = scan_dataset(TRAIN_JSON_DIR, TRAIN_IMG_DIR)
print(f"Training: {len(train_rows)}개\n")

print("Validation scanning...")
val_rows = scan_dataset(VAL_JSON_DIR, VAL_IMG_DIR)
print(f"Validation: {len(val_rows)}개\n")

# 합치기
all_rows = train_rows + val_rows
print(f"total: {len(all_rows)}개")

# CSV 저장
df = pd.DataFrame(all_rows)
df.to_csv("dataset.csv", index=False, encoding="utf-8")
print("dataset.csv 저장 완료")

Training scanning...
이미지 441241개 인덱싱 완료
JSON 441241개 발견


스캔 중:   0%|          | 0/441241 [00:00<?, ?it/s]

Training: 441241개

Validation scanning...
이미지 55155개 인덱싱 완료
JSON 55155개 발견


스캔 중:   0%|          | 0/55155 [00:00<?, ?it/s]

Validation: 55155개

total: 496396개
dataset.csv 저장 완료


In [14]:
df["main_class"] = df["classes"].apply(
    lambda x: CLASS_MAP.get(x.split(",")[0].strip(), -1)
)
df = df[df["main_class"] != -1].reset_index(drop=True)

print("클래스별 분포:")
print(df["main_class"].value_counts())
print(f"\n전체: {len(df)}장")

class_images = defaultdict(set)
for _, row in df.iterrows():
    for cls in row["classes"].split(","):
        cls = cls.strip()
        if cls in CLASS_MAP:
            unified_id = CLASS_MAP[cls]
            class_images[unified_id].add(row["json_path"])

print("통합 클래스별 이미지 수:")
for cls_id, imgs in sorted(class_images.items()):
    status = "전량사용" if len(imgs) <= 5000 else "5000장 샘플링"
    print(f"  클래스 {cls_id}: {len(imgs)}장 → {status}")

selected = set()
for cls_id, imgs in class_images.items():
    sampled = random.sample(list(imgs), min(5000, len(imgs)))
    selected.update(sampled)

subset_df = df[df["json_path"].isin(selected)].reset_index(drop=True)
print(f"\n데이터 추출 완료 → {len(subset_df)}장 추출")

클래스별 분포:
main_class
6     144212
7     126747
5      58145
8      41757
0      34228
3      28996
4      17587
9      17536
2      14531
10      8580
1       4077
Name: count, dtype: int64

전체: 496396장
통합 클래스별 이미지 수:
  클래스 0: 60562장 → 5000장 샘플링
  클래스 1: 13774장 → 5000장 샘플링
  클래스 2: 30038장 → 5000장 샘플링
  클래스 3: 57125장 → 5000장 샘플링
  클래스 4: 19164장 → 5000장 샘플링
  클래스 5: 71097장 → 5000장 샘플링
  클래스 6: 212373장 → 5000장 샘플링
  클래스 7: 212819장 → 5000장 샘플링
  클래스 8: 80945장 → 5000장 샘플링
  클래스 9: 26784장 → 5000장 샘플링
  클래스 10: 14280장 → 5000장 샘플링

데이터 추출 완료 → 52551장 추출


In [15]:
subset_df["main_class"].value_counts()

main_class
6     9024
7     8628
5     5035
3     4913
4     4669
8     4368
0     4082
10    3722
9     3536
2     2965
1     1609
Name: count, dtype: int64

In [16]:
# 분할 먼저 (stratify 적용)
train_df, temp_df = train_test_split(
    subset_df,
    test_size=0.3,
    random_state=RANDOM_SEED,
    stratify=subset_df["main_class"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=temp_df["main_class"]
)
print(f"Train:{len(train_df)} / Val:{len(val_df)} / Test:{len(test_df)}")

def process_one(args):
    row, out_img_dir, out_label_dir = args
    try:
        with open(row["json_path"], encoding="utf-8") as f:
            data = json.load(f)

        img_w = int(row["img_w"])
        img_h = int(row["img_h"])
        name  = Path(row["json_path"]).stem

        lines = []
        for obj in data["objects"]:
            cls_name = obj["class_name"]
            if cls_name not in CLASS_MAP:
                continue
            cls_id = CLASS_MAP[cls_name]
            c  = obj["annotation"]["coord"]
            cx = (c["x"] + c["width"]  / 2) / img_w
            cy = (c["y"] + c["height"] / 2) / img_h
            w  = c["width"]  / img_w
            h  = c["height"] / img_h

            cx = min(max(cx, 0), 1)
            cy = min(max(cy, 0), 1)
            w  = min(max(w,  0), 1)
            h  = min(max(h,  0), 1)

            lines.append(
                f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"
            )

        if not lines:
            return

        with open(f"{out_label_dir}/{name}.txt", "w") as f:
            f.write("\n".join(lines))

        img_array = np.fromfile(row["img_path"], dtype=np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        if img is None:
            return
        img = cv2.resize(img, (640, 640))
        cv2.imwrite(f"{out_img_dir}/{name}.jpg", img)

    except Exception as e:
        print(f"오류: {e}")

def convert_split(df, out_dir, split_name):
    img_dir   = f"{out_dir}/images/{split_name}"
    label_dir = f"{out_dir}/labels/{split_name}"
    os.makedirs(img_dir,   exist_ok=True)
    os.makedirs(label_dir, exist_ok=True)

    args = [
        (row, img_dir, label_dir)
        for _, row in df.iterrows()
    ]

    for arg in tqdm(args, desc=f"{split_name} 변환 중"):
        process_one(arg)

    print(f"{split_name} 완료 → {len(df)}장")

convert_split(train_df, OUTPUT_DIR, "train")
convert_split(val_df,   OUTPUT_DIR, "val")
convert_split(test_df,  OUTPUT_DIR, "test")

Train:36785 / Val:7883 / Test:7883


train 변환 중:   0%|          | 0/36785 [00:00<?, ?it/s]

train 완료 → 36785장


val 변환 중:   0%|          | 0/7883 [00:00<?, ?it/s]

val 완료 → 7883장


test 변환 중:   0%|          | 0/7883 [00:00<?, ?it/s]

test 완료 → 7883장


In [17]:
def create_yaml(out_dir):
    yaml_content = f"""path: ./{out_dir}
train: images/train
val:   images/val
test:  images/test

nc: 11
names:
  0:  종이
  1:  종이팩
  2:  종이컵
  3:  캔류
  4:  재사용유리
  5:  색깔유리
  6:  페트
  7:  플라스틱
  8:  비닐
  9:  스티로폼
  10: 건전지
"""
    with open("data.yaml", "w", encoding="utf-8") as f:
        f.write(yaml_content)
    print("data.yaml 생성 완료")

create_yaml(OUTPUT_DIR)
print("\n전처리 완료")

data.yaml 생성 완료

전처리 완료
